In [2]:

import os
import random
import math
import shutil
import yaml

import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from scipy.ndimage import gaussian_filter
from ultralytics import YOLO
from torchvision import transforms
from typing import List, Tuple
from tqdm import tqdm



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")


WEIGHTS = 'weights/localisateur.pt'

TEST = True

SEUIL_CONF_CLASS    = 0.38
SEUIL_CONF_LOCAL_LO = 0.2   # si classifieur dit POSITIF
SEUIL_RATTRAPAGE    = 0.86  # si classifieur dit NÉGATIF
YOLO_IMG_SIZE = 640   # résolution d'entraînement YOLO
BOX_FRAC = 0.05
CLASSIF_PATH="weights/classifieur.pth"

# ─────────────────────────────────────────────
# 2. PREPROCESSING
# ─────────────────────────────────────────────
def thorax_mask(shape, margin=0.97, blur=51):
    h, w = shape
    y, x = np.ogrid[:h, :w]
    cy, cx = h / 2, w / 2
    ry, rx = (h / 2) * margin, (w / 2) * margin
    mask = ((y - cy) ** 2) / (ry ** 2) + ((x - cx) ** 2) / (rx ** 2)
    mask = (mask <= 1).astype(np.float32)
    mask = cv2.GaussianBlur(mask, (blur, blur), 0)
    return mask


def preprocess_image(img: np.ndarray, clip_percentiles=(1, 99)) -> np.ndarray:
    """Normalise, applique un masque thoracique, CLAHE et renvoie une image RGB uint8."""
    if len(img.shape) == 3:
        img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

    img = img.astype(np.float32)
    p_low, p_high = np.percentile(img, clip_percentiles)
    img = np.clip(img, p_low, p_high)

    denom = p_high - p_low if (p_high - p_low) != 0 else 1e-6
    img = (img - p_low) / denom

    mask = thorax_mask(img.shape)
    background = np.percentile(img, 5)
    img = img * mask + background * (1 - mask)

    img = (img * 255).astype(np.uint8)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    img = clahe.apply(img)
    img = cv2.GaussianBlur(img, (3, 3), sigmaX=0.5)
    return cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)


# ─────────────────────────────────────────────
# 3. SEPE PREPROCESSOR
# ─────────────────────────────────────────────
class SEPEPreprocessor:
    def __init__(self, sigma=1.0, alpha=1.5, sigma_e=0.1, lambda_reg=0.1):
        self.sigma = sigma
        self.alpha = alpha
        self.sigma_e = sigma_e
        self.lambda_reg = lambda_reg

    def __call__(self, image: np.ndarray) -> np.ndarray:
        I = image.astype(np.float32)
        G_sigma = gaussian_filter(I, sigma=self.sigma)
        grad_x = cv2.Sobel(I, cv2.CV_32F, 1, 0, ksize=3)
        grad_y = cv2.Sobel(I, cv2.CV_32F, 0, 1, ksize=3)
        grad_magnitude = np.sqrt(grad_x ** 2 + grad_y ** 2)
        W = np.exp(-(grad_magnitude ** 2) / (self.sigma_e ** 2))
        I_enh = I + self.alpha * W * (I - G_sigma)
        laplacian = cv2.Laplacian(I, cv2.CV_32F)
        I_SEPE = I_enh - self.lambda_reg * np.abs(laplacian)
        return np.clip(I_SEPE, I.min(), I.max())

    def batch_process(self, images: torch.Tensor) -> torch.Tensor:
        c = images.shape[1]
        kernel_size = int(6 * self.sigma) | 1
        x = torch.arange(kernel_size, dtype=torch.float32, device=images.device) - kernel_size // 2
        gauss = torch.exp(-x ** 2 / (2 * self.sigma ** 2))
        gauss = gauss / gauss.sum()
        kernel_2d = (gauss.unsqueeze(0) * gauss.unsqueeze(1)).expand(c, 1, -1, -1)
        padding = kernel_size // 2
        G_sigma = F.conv2d(images, kernel_2d, padding=padding, groups=c)

        sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]],
                                dtype=torch.float32, device=images.device).view(1, 1, 3, 3).expand(c, 1, -1, -1)
        sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]],
                                dtype=torch.float32, device=images.device).view(1, 1, 3, 3).expand(c, 1, -1, -1)
        grad_x = F.conv2d(images, sobel_x, padding=1, groups=c)
        grad_y = F.conv2d(images, sobel_y, padding=1, groups=c)
        grad_mag = torch.sqrt(grad_x ** 2 + grad_y ** 2 + 1e-8)
        W = torch.exp(-(grad_mag ** 2) / (self.sigma_e ** 2))
        I_enh = images + self.alpha * W * (images - G_sigma)

        lap_k = torch.tensor([[0, 1, 0], [1, -4, 1], [0, 1, 0]],
                              dtype=torch.float32, device=images.device).view(1, 1, 3, 3).expand(c, 1, -1, -1)
        laplacian = F.conv2d(images, lap_k, padding=1, groups=c)
        return I_enh - self.lambda_reg * torch.abs(laplacian)


def _build_3ch_batch(images: torch.Tensor, sepe: SEPEPreprocessor) -> torch.Tensor:
    c1 = images[:, 0:1]
    c3 = images[:, 1:2]
    c2 = sepe.batch_process(c1)
    c2_max = c2.view(c2.size(0), -1).max(dim=1)[0].view(-1, 1, 1, 1)
    c2 = c2 / (c2_max + 1e-8)
    return torch.cat([c1, c2, c3], dim=1)


# ─────────────────────────────────────────────
# 4. ARCHITECTURE CLASSIFIEUR
# ─────────────────────────────────────────────
class MEAM(nn.Module):
    def __init__(self, channels: List[int], d_k: int = 64, d_v: int = 64):
        super().__init__()
        self.scales = len(channels)
        self.d_k = d_k
        self.q_proj = nn.ModuleList([nn.Conv2d(c, d_k, 1) for c in channels])
        self.k_proj = nn.ModuleList([nn.Conv2d(c, d_k, 1) for c in channels])
        self.v_proj = nn.ModuleList([nn.Conv2d(c, d_v, 1) for c in channels])
        self.output_proj = nn.Conv2d(d_v * self.scales, channels[-1], 1)
        self.norm = nn.BatchNorm2d(channels[-1])

    def forward(self, features: List[torch.Tensor]) -> torch.Tensor:
        target_size = features[-1].shape[2:]
        resized = [F.interpolate(f, size=target_size, mode='bilinear', align_corners=False)
                   if f.shape[2:] != target_size else f for f in features]
        Q = [p(f) for p, f in zip(self.q_proj, resized)]
        K = [p(f) for p, f in zip(self.k_proj, resized)]
        V = [p(f) for p, f in zip(self.v_proj, resized)]
        attended = []
        for i in range(self.scales):
            scores = [torch.bmm(Q[i].flatten(2).transpose(1, 2), K[j].flatten(2)) / math.sqrt(self.d_k)
                      for j in range(self.scales)]
            weights = [torch.softmax(s, dim=-1) for s in scores]
            agg = sum(torch.bmm(V[j].flatten(2), weights[j].transpose(1, 2)) for j in range(self.scales))
            attended.append(agg.view_as(V[i]))
        fused = torch.cat(attended, dim=1)
        return F.relu(self.norm(self.output_proj(fused)))


class EfficientBackbone(nn.Module):
    EXTRACT_LAYERS = [2, 3, 4, 6]
    _CHANNEL_MAP = {'s': [48, 64, 128, 256], 'm': [48, 80, 160, 304], 'l': [64, 96, 192, 384]}

    def __init__(self, variant='s', pretrained=True):
        super().__init__()
        self.extract_layers = self.EXTRACT_LAYERS
        self.channels = self._CHANNEL_MAP[variant]
        if variant == 's':
            from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights
            weights = EfficientNet_V2_S_Weights.IMAGENET1K_V1 if pretrained else None
            self.backbone = efficientnet_v2_s(weights=weights).features
        elif variant == 'm':
            from torchvision.models import efficientnet_v2_m, EfficientNet_V2_M_Weights
            weights = EfficientNet_V2_M_Weights.IMAGENET1K_V1 if pretrained else None
            self.backbone = efficientnet_v2_m(weights=weights).features
        else:
            from torchvision.models import efficientnet_v2_l, EfficientNet_V2_L_Weights
            weights = EfficientNet_V2_L_Weights.IMAGENET1K_V1 if pretrained else None
            self.backbone = efficientnet_v2_l(weights=weights).features

    def forward(self, x):
        features = []
        for i, block in enumerate(self.backbone):
            x = block(x)
            if i in self.extract_layers:
                features.append(x)
        return features, features[-1]
def _letterbox(img_rgb: np.ndarray, target: int) -> Tuple[np.ndarray, float, int, int]:
    """
    Redimensionne img_rgb dans un carré target×target avec letterboxing gris.
    Retourne (img_resized, scale, pad_x, pad_y).
    """
    h, w = img_rgb.shape[:2]
    scale = min(target / w, target / h)
    new_w = int(round(w * scale))
    new_h = int(round(h * scale))
    resized = cv2.resize(img_rgb, (new_w, new_h), interpolation=cv2.INTER_LINEAR)

    canvas = np.full((target, target, 3), 114, dtype=np.uint8)
    pad_x = (target - new_w) // 2
    pad_y = (target - new_h) // 2
    canvas[pad_y:pad_y + new_h, pad_x:pad_x + new_w] = resized
    return canvas, scale, pad_x, pad_y

class NoduleClassifier(nn.Module):
    def __init__(self, backbone_variant='s'):
        super().__init__()
        self.backbone = EfficientBackbone(variant=backbone_variant)
        self.meam = MEAM(channels=self.backbone.channels)
        final_channels = self.backbone.channels[-1]
        self.classifier_head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(final_channels, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        features_list, _ = self.backbone(x)
        fused = self.meam(features_list)
        return self.classifier_head(fused)


# ─────────────────────────────────────────────
# 5. WRAPPERS CLASSIFIEUR & LOCALISATEUR
# ─────────────────────────────────────────────
class Classifieur:
    def __init__(self, model_path=CLASSIF_PATH, device=device):
        self.device = device
        self.model = NoduleClassifier()
        state_dict = torch.load(model_path, map_location=device, weights_only=False)
        self.model.load_state_dict(state_dict)
        self.model.to(device).eval()

    def preprocess(self, img_np: np.ndarray) -> torch.Tensor:
        if len(img_np.shape) == 3 and img_np.shape[2] == 3:
            img_gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
        elif len(img_np.shape) == 3:
            img_gray = img_np[:, :, 0]
        else:
            img_gray = img_np

        if img_gray.dtype != np.uint8:
            mn, mx = img_gray.min(), img_gray.max()
            img_gray = ((img_gray - mn) / (mx - mn + 1e-8) * 255).astype(np.uint8)

        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        img_clahe = clahe.apply(img_gray)
        t_raw   = torch.from_numpy(img_gray).float() / 255.0
        t_clahe = torch.from_numpy(img_clahe).float() / 255.0
        t_input = torch.stack([t_raw, t_clahe], dim=0).unsqueeze(0).to(self.device)
        return _build_3ch_batch(t_input, SEPEPreprocessor())

    def pred(self, img_np: np.ndarray, threshold: float = SEUIL_CONF_CLASS) -> torch.Tensor:
        img_tensor = self.preprocess(img_np)
        with torch.no_grad():
            outputs = self.model(img_tensor)
            probs = F.softmax(outputs, dim=1)
            return (probs[:, 1] >= threshold).long()


class Localisation:
    def __init__(self, model_path: str, device=device):
        self.model = YOLO(model_path).to(device)

    def pred(self, img_np: np.ndarray, conf_threshold: float = 0.50) -> List[dict]:
        img_rgb = preprocess_image(img_np)
        # Letterbox explicite — cohérent avec le dataset d'entraînement
        img_lb, scale, pad_x, pad_y = _letterbox(img_rgb, YOLO_IMG_SIZE)
        h_orig, w_orig = img_rgb.shape[:2]

        # Bornes de la zone image réelle dans le canvas letterboxé (hors padding gris)
        x_min_v = pad_x
        y_min_v = pad_y
        x_max_v = pad_x + int(round(w_orig * scale))
        y_max_v = pad_y + int(round(h_orig * scale))

        # Image en niveaux de gris pour le filtre anatomique (colonne/os)
        img_gray_lb = cv2.cvtColor(img_lb, cv2.COLOR_RGB2GRAY).astype(np.float32)

        results = self.model(img_lb, conf=conf_threshold, verbose=False)
        nodules = []
        for box in results[0].boxes:
            conf = float(box.conf[0])
            if conf < conf_threshold:
                continue
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            cx = (x1 + x2) / 2.0
            cy = (y1 + y2) / 2.0

            # Rejette les détections dont le centre tombe dans la zone de padding
            if not (x_min_v <= cx <= x_max_v and y_min_v <= cy <= y_max_v):
                continue

            # ── Filtre anatomique : colonne vertébrale et os ─────────────────
            # La colonne est une bande verticale très lumineuse au centre.
            # Les os (côtes, clavicules) sont aussi très intenses.
            # On rejette les détections dont la région est trop lumineuse.
            rx1, ry1 = int(max(0, x1)), int(max(0, y1))
            rx2, ry2 = int(min(YOLO_IMG_SIZE, x2)), int(min(YOLO_IMG_SIZE, y2))
            if rx2 > rx1 and ry2 > ry1:
                patch = img_gray_lb[ry1:ry2, rx1:rx2]
                mean_intensity = float(patch.mean())
                # Seuil empirique : pixels > 200/255 = os/colonne (très blanc après CLAHE)
                if mean_intensity > 200:
                    continue

            # ── Reprojection vers l'espace image originale ───────────────────
            # Les coordonnées YOLO sont dans l'espace letterbox (0–640).
            # GT dans df_bbox est en pixels originaux → on reprojecte.
            cx_orig = (cx - pad_x) / scale
            cy_orig = (cy - pad_y) / scale

            nodules.append({
                'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2,
                'cx': cx_orig,   # espace image originale
                'cy': cy_orig,   # espace image originale
                'cx_lb': cx,     # espace letterbox (utile pour visualisation)
                'cy_lb': cy,
                'conf': conf
            })
        return nodules



Device : cuda


In [3]:
class Model:
    def __init__(
        self,
        classif_path: str = CLASSIF_PATH,
        weight: str = WEIGHTS,
        device=device,
    ):
        self.classifieur   = Classifieur(model_path=classif_path, device=device)
        self.localisateur  = Localisation(model_path=weight, device=device)

    def predict(self, img_np: np.ndarray) -> dict:
        """
        Pipeline de prédiction :
          1. Classifieur  →  POSITIF / NÉGATIF  (seuil SEUIL_CONF_CLASS)
          2. Si NÉGATIF   →  double-vérif localisateur (seuil SEUIL_RATTRAPAGE)
          3. Si l'un des deux est POSITIF  →  localisation fine (seuil SEUIL_CONF_LOCAL_LO)
          4. Sinon        →  pas de nodule, liste vide

        Retourne
        --------
        {
            "positive"    : bool,
            "source"      : "classifier" | "fallback_localizer" | "negative",
            "nodules"     : List[dict]   # vide si négatif
        }
        """
        # ── étape 1 : classifieur ────────────────────────────────────────────
        pred_class = self.classifieur.pred(img_np, threshold=SEUIL_CONF_CLASS).item()
        positive = bool(pred_class)
        source   = "classifier" if positive else None

        # ── étape 2 : rattrapage par le localisateur si classifieur dit NON ──
        if not positive:
            fallback = self.localisateur.pred(img_np, conf_threshold=SEUIL_RATTRAPAGE)
            if fallback:
                positive = True
                source   = "fallback_localizer"

        # ── étape 3 : localisation fine si au moins un vote POSITIF ──────────
        if positive:
            nodules = self.localisateur.pred(img_np, conf_threshold=SEUIL_CONF_LOCAL_LO)
            # Un rattrapage peut avoir voté OUI alors que la localisation fine
            # ne trouve rien au seuil bas → on considère quand même positif
            # (le classifieur ou le rattrapage l'a confirmé).
            return {"positive": True, "source": source, "nodules": nodules}

        # ── étape 4 : négatif confirmé ────────────────────────────────────────
        return {"positive":positive, "source": source, "nodules": []}

In [4]:
CSV_BBOX        = "localization_labels.csv"        # x, y centre nodule
CSV_LABELS      = "classification_labels.csv" 

In [5]:
def display_prediction(img_np: np.ndarray,
                       result: dict,
                       gt_pts: list = None,
                       save_path: str = "prediction.png") -> None:
    """
    Affiche une image avec les détections du pipeline.
 
    img_np   : image brute (entrée du modèle)
    result   : dict retourné par Model.predict()
    gt_pts   : liste optionnelle de {"x": int, "y": int} en pixels originaux (GT)
    save_path: chemin de sauvegarde du PNG
    """
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
 
    # Preprocessing + letterbox identique à Localisation.pred
    img_rgb = preprocess_image(img_np)
    img_lb, scale, pad_x, pad_y = _letterbox(img_rgb, YOLO_IMG_SIZE)
 
    fig, ax = plt.subplots(1, 1, figsize=(8, 8), facecolor="#111111")
    ax.imshow(img_lb)
    ax.axis("off")
 
    # ── GT nodules (croix rouge) ─────────────────────────────────────────────
    if gt_pts:
        for pt in gt_pts:
            cx_lb = pt["x"] * scale + pad_x
            cy_lb = pt["y"] * scale + pad_y
            s = 22
            ax.plot([cx_lb - s, cx_lb + s], [cy_lb,     cy_lb    ], color="#ff3333", lw=2.5)
            ax.plot([cx_lb,     cx_lb    ], [cy_lb - s, cy_lb + s], color="#ff3333", lw=2.5)
 
    # ── Prédictions YOLO (cercle cyan + conf) ───────────────────────────────
    for n in result["nodules"]:
        # cx_lb / cy_lb sont les coords letterbox stockées dans le dict
        cx_lb = n.get("cx_lb", n["cx"] * scale + pad_x)
        cy_lb = n.get("cy_lb", n["cy"] * scale + pad_y)
        conf  = n["conf"]
        circ  = plt.Circle((cx_lb, cy_lb), radius=18, fill=False,
                            edgecolor="#00e5ff", linewidth=2.0)
        ax.add_patch(circ)
        ax.text(cx_lb + 22, cy_lb - 10, f"{conf:.2f}",
                color="#00e5ff", fontsize=8, fontweight="bold")
 
    # ── Titre ────────────────────────────────────────────────────────────────
    status_color = "#aaffaa" if result["positive"] else "#ff6666"
    status_txt   = "POSITIF" if result["positive"] else "NÉGATIF"
    source_txt   = result["source"]
    n_det        = len(result["nodules"])
    title = f"{status_txt}  |  source: {source_txt}  |  {n_det} nodule(s) détecté(s)"
    ax.set_title(title, fontsize=11, color=status_color, pad=6)
 
    # ── Légende ──────────────────────────────────────────────────────────────
    legend = []
    if gt_pts:
        legend.append(mpatches.Patch(color="#ff3333", label="GT nodule"))
    if result["nodules"]:
        legend.append(mpatches.Patch(color="#00e5ff", label="Prédiction YOLO"))
    if legend:
        ax.legend(handles=legend, loc="lower right", fontsize=8,
                  framealpha=0.7, facecolor="#222222", labelcolor="white")
 
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.close()
    print(f"  → Image sauvegardée : {save_path}")
    print(f"  → Positive : {result['positive']}  |  Source : {result['source']}")
    for i, n in enumerate(result["nodules"]):
        print(f"     Nodule {i+1} : cx={n['cx']:.1f}px  cy={n['cy']:.1f}px  conf={n['conf']:.3f}")

In [6]:
def _load_image_from_disk(path_img: str) -> np.ndarray:
    """Charge une image depuis le disque (path relatif au répertoire courant)."""
    img = Image.open(path_img)
    return np.array(img)

In [7]:
def load_data(csv_bbox=CSV_BBOX, csv_labels=CSV_LABELS):
    df_bbox = pd.read_csv(csv_bbox)

    labels = pd.read_csv(csv_labels)
    labels["label"] = (labels["label"] != "No Finding").astype(int)
    labels["path"] = np.where(
        labels["LIDC_ID"].isna(),
        "nih_filtered_images/",
        "lidc_png_16_bit/"
    ) + labels["file_name"]

    no_sane  = labels[labels["label"] == 1]
    annoted  = labels[labels["LIDC_ID"].notna()]
    return df_bbox, labels, no_sane, annoted

df_bbox, labels, no_sane, annoted = load_data()

In [8]:
model = Model(weight=WEIGHTS)

img = _load_image_from_disk("dataset_extrait/lidc_png_16_bit/0008.png") ##,"nih_filtered_images/00018592_001.png"
result = model.predict(img)

# Sans GT
display_prediction(img, result, save_path="prediction_0008.png")

# Avec GT (si tu as les coords dans df_bbox)
gt = df_bbox[df_bbox["file_name"] == "0008.png"]
gt_pts = [{"x": r["x"], "y": r["y"]} for _, r in gt.iterrows()]
display_prediction(img, result, gt_pts=gt_pts, save_path="prediction_0008.png")

  → Image sauvegardée : prediction_0008.png
  → Positive : True  |  Source : classifier
     Nodule 1 : cx=1483.4px  cy=742.3px  conf=0.844
     Nodule 2 : cx=476.7px  cy=1033.5px  conf=0.727
  → Image sauvegardée : prediction_0008.png
  → Positive : True  |  Source : classifier
     Nodule 1 : cx=1483.4px  cy=742.3px  conf=0.844
     Nodule 2 : cx=476.7px  cy=1033.5px  conf=0.727


In [ ]:
for 

In [9]:
import argparse
from pathlib import Path

def run_inference(input_dir: str, output_dir: str):
    model = Model()
    input_path = Path(input_dir)
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    classif_data = []
    localiz_data = []

    extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff')
    image_files = [f for f in input_path.iterdir() if f.suffix.lower() in extensions]

    for img_file in tqdm(image_files, desc="Inference"):
        img = cv2.imread(str(img_file))
        if img is None:
            continue
        
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        result = model.predict(img_rgb)

        label = "Nodule" if result["positive"] else "No Finding"
        
        conf_class = 1.0
        if result["nodules"]:
            conf_class = max([n['conf'] for n in result["nodules"]])
        elif not result["positive"]:
            conf_class = 1.0

        classif_data.append({
            "file_name": img_file.name,
            "label": label,
            "confidence": conf_class
        })

        if result["positive"] and result["nodules"]:
            for nodule in result["nodules"]:
                localiz_data.append({
                    "file_name": img_file.name,
                    "x": nodule["cx"],
                    "y": nodule["cy"],
                    "confidence": nodule["conf"]
                })

    df_classif = pd.DataFrame(classif_data)
    df_classif.to_csv(output_path / "classification_test_results.csv", index=False)

    df_localiz = pd.DataFrame(localiz_data)
    df_localiz.to_csv(output_path / "localization_test_results.csv", index=False)

